# SmartPlay AI Recommendation System
## Data Preprocessing

This notebook performs preprocessing of the raw datasets used for the personalized learning-path recommendation component.

### Objectives

- Load the raw datasets
- Remove invalid and empty records
- Check and handle missing values
- Remove duplicate records
- Standardize categorical values
- Convert data types where required
- Transform selected fields into machine-learning-ready formats
- Save the processed datasets as CSV files

In [112]:
import pandas as pd
import numpy as np
import os
import re

print("Current working directory:")
print(os.getcwd())

Current working directory:
/Users/aathilhakam/Desktop/Research/An-AI-Driven-Teacher-Assisted-Learning-System/SmartPlay-AI-Recommendation/notebooks


In [113]:
print(os.path.exists("../data/raw/original_files/original_school_data"))
print(os.path.exists("../data/raw/original_files/ai_learning_path"))
print(os.path.exists("../data/processed"))

True
True
True


In [114]:
import os

RAW_SCHOOL_PATH = "../data/raw/original_files/original_school_data"
RAW_AI_PATH = "../data/raw/original_files/ai_learning_path"

PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Raw school data:", RAW_SCHOOL_PATH)
print("Raw AI learning path:", RAW_AI_PATH)
print("Processed data:", PROCESSED_PATH)

Raw school data: ../data/raw/original_files/original_school_data
Raw AI learning path: ../data/raw/original_files/ai_learning_path
Processed data: ../data/processed


In [115]:
import pandas as pd

student_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "1. Student Basic Information (Anonymized).xlsx"
    )
)

curriculum_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "2. Curriculum and Lesson Data.xlsx"
    )
)

activity_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "3. Activity Performance Data.xlsx"
    )
)

assessment_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "4. Assessment and Test Results.xlsx"
    )
)

teacher_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "5. Teacher Observation Data (Teacher feedback ).xlsx"
    )
)

historical_df = pd.read_excel(
    os.path.join(
        RAW_SCHOOL_PATH,
        "6. Historical Learning Path Data.xlsx"
    )
)

ai_learning_df = pd.read_excel(
    os.path.join(
        RAW_AI_PATH,
        "Grade_1_AI_Learning_Path_Dataset.xlsx"
    )
)

print("All 7 datasets loaded successfully.")

All 7 datasets loaded successfully.


In [116]:
datasets = {
    "Student": student_df,
    "Curriculum": curriculum_df,
    "Activity": activity_df,
    "Assessment": assessment_df,
    "Teacher Observation": teacher_df,
    "Historical Learning Path": historical_df,
    "AI Learning Path": ai_learning_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Student: (100, 8)
Curriculum: (21, 9)
Activity: (60, 12)
Assessment: (30, 15)
Teacher Observation: (100, 9)
Historical Learning Path: (60, 6)
AI Learning Path: (11, 14)


In [117]:
#Check missing values

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Missing values:")
    print(df.isnull().sum())


Student
Missing values:
Student_ID              0
Student_Name            0
Age                     0
Gender                  0
School Name             0
District                0
Grade                   0
Learning_Environment    0
dtype: int64

Curriculum
Missing values:
Lesson_ID               1
Grade                   1
Subject                 1
Topic                   1
Competency              1
Learning_Objective      1
Difficulty_Level        1
Required_Skills         1
Classroom_Activities    1
dtype: int64

Activity
Missing values:
Activity_ID            0
Student_ID             0
Lesson_ID              0
Activity_Type          0
Difficulty             0
Date                   0
Completion_Status      0
Attempts               0
Time_Taken_Min         0
Score                  0
Errors                25
Teacher_Evaluation     0
dtype: int64

Assessment
Missing values:
Assessment_ID             0
Student_ID                0
Subject                   0
Assessment_Type           0


In [118]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    missing = df.isnull().sum()

    for column, count in missing.items():
        if count > 0:
            print(f"{column}: {count}")


Student

Curriculum
Lesson_ID: 1
Grade: 1
Subject: 1
Topic: 1
Competency: 1
Learning_Objective: 1
Difficulty_Level: 1
Required_Skills: 1
Classroom_Activities: 1

Activity
Errors: 25

Assessment

Teacher Observation

Historical Learning Path

AI Learning Path


In [119]:
#Curriculum — remove the empty row

curriculum_clean = curriculum_df.dropna(how="all").copy()

print("Before:", curriculum_df.shape)
print("After:", curriculum_clean.shape)

Before: (21, 9)
After: (20, 9)


In [120]:
activity_clean = activity_df.copy()

activity_clean["Errors"] = activity_clean["Errors"].fillna(
    "None recorded"
)

print(activity_clean["Errors"].isnull().sum())

0


In [121]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)


Student
No missing values

Curriculum
Lesson_ID               1
Grade                   1
Subject                 1
Topic                   1
Competency              1
Learning_Objective      1
Difficulty_Level        1
Required_Skills         1
Classroom_Activities    1
dtype: int64

Activity
Errors    25
dtype: int64

Assessment
No missing values

Teacher Observation
No missing values

Historical Learning Path
No missing values

AI Learning Path
No missing values


In [122]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [123]:
for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing.to_string())


Student
No missing values

Curriculum
Lesson_ID               1
Grade                   1
Subject                 1
Topic                   1
Competency              1
Learning_Objective      1
Difficulty_Level        1
Required_Skills         1
Classroom_Activities    1

Activity
Errors    25

Assessment
No missing values

Teacher Observation
No missing values

Historical Learning Path
No missing values

AI Learning Path
No missing values


In [124]:
#Check Assessment

assessment_missing = assessment_df.isnull().sum()
assessment_missing = assessment_missing[assessment_missing > 0]

print("Assessment missing values:")

if len(assessment_missing) == 0:
    print("No missing values")
else:
    print(assessment_missing.to_string())

Assessment missing values:
No missing values


In [125]:
#Check Teacher Observation

teacher_missing = teacher_df.isnull().sum()
teacher_missing = teacher_missing[teacher_missing > 0]

print("Teacher Observation missing values:")

if len(teacher_missing) == 0:
    print("No missing values")
else:
    print(teacher_missing.to_string())

Teacher Observation missing values:
No missing values


In [126]:
#Check Historical Learning Path

historical_missing = historical_df.isnull().sum()
historical_missing = historical_missing[historical_missing > 0]

print("Historical Learning Path missing values:")

if len(historical_missing) == 0:
    print("No missing values")
else:
    print(historical_missing.to_string())

Historical Learning Path missing values:
No missing values


In [127]:
#Check AI Learning Path

ai_missing = ai_learning_df.isnull().sum()
ai_missing = ai_missing[ai_missing > 0]

print("AI Learning Path missing values:")

if len(ai_missing) == 0:
    print("No missing values")
else:
    print(ai_missing.to_string())

AI Learning Path missing values:
No missing values


## 2. Missing Value Handling

The exploratory analysis identified missing values in two datasets.

- The Curriculum dataset contains one completely empty row, which is removed.
- The Activity Performance dataset contains missing values in the Errors column. These are treated as "None recorded" because the activity record itself is valid and the missing value indicates that no error was recorded.
- The remaining datasets contain no missing values.

In [128]:
# Create copies of the original datasets
student_clean = student_df.copy()
curriculum_clean = curriculum_df.copy()
activity_clean = activity_df.copy()
assessment_clean = assessment_df.copy()
teacher_clean = teacher_df.copy()
historical_clean = historical_df.copy()
ai_learning_clean = ai_learning_df.copy()

print("Clean copies created successfully.")

Clean copies created successfully.


In [129]:
# Remove completely empty rows(Curriculum)
curriculum_clean = curriculum_clean.dropna(how="all").reset_index(drop=True)

print("Original Curriculum shape:", curriculum_df.shape)
print("Cleaned Curriculum shape:", curriculum_clean.shape)

Original Curriculum shape: (21, 9)
Cleaned Curriculum shape: (20, 9)


In [130]:
# Replace missing error descriptions(Activity Performance)
activity_clean["Errors"] = activity_clean["Errors"].fillna("None recorded")

print("Missing Errors after cleaning:")
print(activity_clean["Errors"].isnull().sum())

Missing Errors after cleaning:
0


In [131]:
cleaned_datasets = {
    "Student": student_clean,
    "Curriculum": curriculum_clean,
    "Activity": activity_clean,
    "Assessment": assessment_clean,
    "Teacher Observation": teacher_clean,
    "Historical Learning Path": historical_clean,
    "AI Learning Path": ai_learning_clean
}

for name, df in cleaned_datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())


Student
Shape: (100, 8)
Missing values: 0
Duplicate rows: 0

Curriculum
Shape: (20, 9)
Missing values: 0
Duplicate rows: 0

Activity
Shape: (60, 12)
Missing values: 0
Duplicate rows: 0

Assessment
Shape: (30, 15)
Missing values: 0
Duplicate rows: 0

Teacher Observation
Shape: (100, 9)
Missing values: 0
Duplicate rows: 0

Historical Learning Path
Shape: (60, 6)
Missing values: 0
Duplicate rows: 0

AI Learning Path
Shape: (11, 14)
Missing values: 0
Duplicate rows: 0


In [132]:
print("Student:", student_df.shape)
print("Curriculum:", curriculum_df.shape)
print("Activity:", activity_df.shape)
print("Assessment:", assessment_df.shape)
print("Teacher:", teacher_df.shape)
print("Historical:", historical_df.shape)
print("AI Learning Path:", ai_learning_df.shape)

Student: (100, 8)
Curriculum: (21, 9)
Activity: (60, 12)
Assessment: (30, 15)
Teacher: (100, 9)
Historical: (60, 6)
AI Learning Path: (11, 14)


In [133]:
# Recreate clean copies

student_clean = student_df.copy()
curriculum_clean = curriculum_df.copy()
activity_clean = activity_df.copy()
assessment_clean = assessment_df.copy()
teacher_clean = teacher_df.copy()
historical_clean = historical_df.copy()
ai_learning_clean = ai_learning_df.copy()

print("Clean copies recreated successfully.")

Clean copies recreated successfully.


In [134]:
# Remove completely empty rows from Curriculum

curriculum_clean = (
    curriculum_clean
    .dropna(how="all")
    .reset_index(drop=True)
)

print("Original Curriculum:", curriculum_df.shape)
print("Cleaned Curriculum:", curriculum_clean.shape)

Original Curriculum: (21, 9)
Cleaned Curriculum: (20, 9)


In [135]:
# Handle missing error descriptions in Activity Performance

activity_clean["Errors"] = activity_clean["Errors"].fillna(
    "None recorded"
)

print(
    "Missing Errors after cleaning:",
    activity_clean["Errors"].isnull().sum()
)

Missing Errors after cleaning: 0


In [136]:
cleaned_datasets = {
    "Student": student_clean,
    "Curriculum": curriculum_clean,
    "Activity": activity_clean,
    "Assessment": assessment_clean,
    "Teacher Observation": teacher_clean,
    "Historical Learning Path": historical_clean,
    "AI Learning Path": ai_learning_clean
}

for name, df in cleaned_datasets.items():
    print(
        f"{name}: "
        f"Shape={df.shape}, "
        f"Missing={df.isnull().sum().sum()}, "
        f"Duplicates={df.duplicated().sum()}"
    )

Student: Shape=(100, 8), Missing=0, Duplicates=0
Curriculum: Shape=(20, 9), Missing=0, Duplicates=0
Activity: Shape=(60, 12), Missing=0, Duplicates=0
Assessment: Shape=(30, 15), Missing=0, Duplicates=0
Teacher Observation: Shape=(100, 9), Missing=0, Duplicates=0
Historical Learning Path: Shape=(60, 6), Missing=0, Duplicates=0
AI Learning Path: Shape=(11, 14), Missing=0, Duplicates=0


## 3. Data Type Validation

The cleaned datasets are checked to ensure that numerical variables are stored as numerical data types, date variables are stored as datetime values, and categorical variables remain suitable for further preprocessing.

In [137]:
for name, df in cleaned_datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(df.dtypes)


Student
Student_ID              object
Student_Name            object
Age                      int64
Gender                  object
School Name             object
District                object
Grade                   object
Learning_Environment    object
dtype: object

Curriculum
Lesson_ID               object
Grade                   object
Subject                 object
Topic                   object
Competency              object
Learning_Objective      object
Difficulty_Level        object
Required_Skills         object
Classroom_Activities    object
dtype: object

Activity
Activity_ID                   object
Student_ID                    object
Lesson_ID                     object
Activity_Type                 object
Difficulty                    object
Date                  datetime64[ns]
Completion_Status             object
Attempts                       int64
Time_Taken_Min                 int64
Score                          int64
Errors                        object
Teacher

## 4. Numerical Feature Conversion

Numerical variables are converted into appropriate numeric data types so that they can be used later during feature engineering and machine learning.

In [138]:
numeric_columns = {
    "Student": [
        "Age"
    ],

    "Activity": [
        "Attempts",
        "Time_Taken_Min",
        "Score"
    ],

    "Assessment": [
        "Pre_Test",
        "Post_Test",
        "Unit_Test",
        "Monthly_Test",
        "Term_Result",
        "Motor_Skills_Score",
        "Creativity_Score",
        "Interactive_Engagement",
        "Learning_Progress"
    ],

    "Historical Learning Path": [
        "Sequence_No"
    ]
}

for dataset_name, columns in numeric_columns.items():

    df = cleaned_datasets[dataset_name]

    for column in columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

print("Numerical columns converted successfully.")

Numerical columns converted successfully.


In [139]:
for dataset_name, columns in numeric_columns.items():

    df = cleaned_datasets[dataset_name]

    print("\n" + "=" * 60)
    print(dataset_name)

    for column in columns:
        print(column, "->", df[column].dtype)


Student
Age -> int64

Activity
Attempts -> int64
Time_Taken_Min -> int64
Score -> int64

Assessment
Pre_Test -> int64
Post_Test -> int64
Unit_Test -> int64
Monthly_Test -> int64
Term_Result -> int64
Motor_Skills_Score -> int64
Creativity_Score -> int64
Interactive_Engagement -> int64
Learning_Progress -> int64

Historical Learning Path
Sequence_No -> int64


## 5. Categorical Data Cleaning

Categorical and text fields are cleaned by removing unnecessary whitespace and ensuring that textual values have a consistent representation. These fields will be encoded later during feature engineering.

In [140]:
# Clean categorical/text columns

for name, df in cleaned_datasets.items():

    object_columns = df.select_dtypes(include=["object"]).columns

    for column in object_columns:
        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
        )

print("Categorical/text columns cleaned successfully.")

Categorical/text columns cleaned successfully.


In [141]:
# Check categorical values

for name, df in cleaned_datasets.items():

    print("\n" + "=" * 70)
    print(name)

    object_columns = df.select_dtypes(include=["object"]).columns

    for column in object_columns:
        print(f"\n{column}:")
        print(df[column].unique()[:10])


Student

Student_ID:
['ST001' 'ST002' 'ST003' 'ST004' 'ST005' 'ST006' 'ST007' 'ST008' 'ST009'
 'ST010']

Student_Name:
['Ahamed Rizwan' 'Aisha Fathima' 'Ayaan Nazeer' 'Fathima Zahra'
 'Ibrahim Faiz' 'Maryam Haniya' 'Rayan Ashraf' 'Hafsa Amani' 'Zayan Imran'
 'Inaya Sufiya']

Gender:
['Male' 'Female']

School Name:
['AK/Iqrah VIdyalaya - Addalaichenai - Home']

District:
['Ampara']

Grade:
['Grade 1']

Learning_Environment:
['Classroom + Digital Learning' 'Classroom Only' 'Home + Classroom'
 'Blended Learning' 'Activity-Based Learning'
 'Teacher Guided Individual Learning' 'Parent Supported Learning'
 'Play-Based Learning']

Curriculum

Lesson_ID:
['LES021' 'LES022' 'LES023' 'LES024' 'LES025' 'LES026' 'LES027' 'LES028'
 'LES029' 'LES030']

Grade:
['Grade 1']

Subject:
['Mathematics' 'English' 'Sinhala' 'Environmental Studies'
 'Health & Physical Education' 'ICT' 'Creative Activities' 'Life Skills']

Topic:
['Number Recognition' 'Simple Words' 'Reading Practice'
 'School Environment' 'N

## 7. Date Processing

The activity performance dataset contains a Date field representing when each learning activity was performed. The date is converted into a datetime format so that temporal information can be used during feature engineering.

In [142]:
# Convert Activity Date to datetime

activity_clean["Date"] = pd.to_datetime(
    activity_clean["Date"],
    errors="coerce"
)

print("Date data type:", activity_clean["Date"].dtype)
print("Missing dates:", activity_clean["Date"].isnull().sum())

Date data type: datetime64[ns]
Missing dates: 0


In [143]:
# ==========================================
# FINAL PREPROCESSING VALIDATION
# ==========================================

print("=" * 70)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 70)

all_passed = True

for name, df in cleaned_datasets.items():

    print(f"\n{'-' * 70}")
    print(f"Dataset: {name}")
    print(f"{'-' * 70}")

    # 1. Dataset shape
    print("Shape:", df.shape)

    # 2. Missing values
    missing_values = df.isnull().sum().sum()
    print("Missing values:", missing_values)

    # 3. Duplicate rows
    duplicate_rows = df.duplicated().sum()
    print("Duplicate rows:", duplicate_rows)

    # 4. Empty rows
    empty_rows = df.isna().all(axis=1).sum()
    print("Completely empty rows:", empty_rows)

    # 5. Data types
    print("\nData types:")
    print(df.dtypes)

    # 6. Validation status
    if missing_values == 0 and duplicate_rows == 0 and empty_rows == 0:
        print("STATUS: PASS")
    else:
        print("STATUS: CHECK REQUIRED")
        all_passed = False

print("\n" + "=" * 70)

if all_passed:
    print("FINAL RESULT: PREPROCESSING SUCCESSFUL")
else:
    print("FINAL RESULT: PREPROCESSING REQUIRES ATTENTION")

print("=" * 70)

FINAL PREPROCESSING VALIDATION

----------------------------------------------------------------------
Dataset: Student
----------------------------------------------------------------------
Shape: (100, 8)
Missing values: 0
Duplicate rows: 0
Completely empty rows: 0

Data types:
Student_ID              object
Student_Name            object
Age                      int64
Gender                  object
School Name             object
District                object
Grade                   object
Learning_Environment    object
dtype: object
STATUS: PASS

----------------------------------------------------------------------
Dataset: Curriculum
----------------------------------------------------------------------
Shape: (20, 9)
Missing values: 0
Duplicate rows: 0
Completely empty rows: 0

Data types:
Lesson_ID               object
Grade                   object
Subject                 object
Topic                   object
Competency              object
Learning_Objective      object
Diffic

In [144]:
# ==========================================
# COLUMN VALIDATION
# ==========================================

expected_columns = {
    "Student": [
        "Student_ID",
        "Student_Name",
        "Age",
        "Gender",
        "School Name",
        "District",
        "Grade",
        "Learning_Environment"
    ],

    "Curriculum": [
        "Lesson_ID",
        "Grade",
        "Subject",
        "Topic",
        "Competency",
        "Learning_Objective",
        "Difficulty_Level",
        "Required_Skills",
        "Classroom_Activities"
    ],

    "Activity": [
        "Activity_ID",
        "Student_ID",
        "Lesson_ID",
        "Activity_Type",
        "Difficulty",
        "Date",
        "Completion_Status",
        "Attempts",
        "Time_Taken_Min",
        "Score",
        "Errors",
        "Teacher_Evaluation"
    ],

    "Assessment": [
        "Assessment_ID",
        "Student_ID",
        "Subject",
        "Assessment_Type",
        "Pre_Test",
        "Post_Test",
        "Unit_Test",
        "Monthly_Test",
        "Term_Result",
        "Competency_Score",
        "Motor_Skills_Score",
        "Creativity_Score",
        "Interactive_Engagement",
        "Learning_Progress",
        "Recommended_Action"
    ]
}

print("=" * 70)
print("COLUMN VALIDATION")
print("=" * 70)

for dataset_name, columns in expected_columns.items():

    df = cleaned_datasets[dataset_name]

    missing_columns = [
        column for column in columns
        if column not in df.columns
    ]

    if len(missing_columns) == 0:
        print(f"{dataset_name}: PASS")
    else:
        print(f"{dataset_name}: CHECK")
        print("Missing columns:", missing_columns)

COLUMN VALIDATION
Student: PASS
Curriculum: PASS
Activity: PASS
Assessment: PASS


In [145]:
# ==========================================
# DATE VALIDATION
# ==========================================

print("=" * 70)
print("DATE VALIDATION")
print("=" * 70)

print("Date data type:", activity_clean["Date"].dtype)
print("Missing dates:", activity_clean["Date"].isnull().sum())

if pd.api.types.is_datetime64_any_dtype(activity_clean["Date"]):
    print("Date conversion: PASS")
else:
    print("Date conversion: CHECK")

DATE VALIDATION
Date data type: datetime64[ns]
Missing dates: 0
Date conversion: PASS


In [146]:
# ==========================================
# PREPROCESSING FINAL SUMMARY
# ==========================================

print("\n" + "=" * 70)
print("PREPROCESSING FINAL SUMMARY")
print("=" * 70)

for name, df in cleaned_datasets.items():

    print(
        f"{name:<25} "
        f"Rows={len(df):<5} "
        f"Columns={len(df.columns):<5} "
        f"Missing={df.isnull().sum().sum():<3} "
        f"Duplicates={df.duplicated().sum():<3}"
    )

print("=" * 70)
print("Preprocessing stage completed.")
print("Data is ready for Feature Engineering.")
print("=" * 70)


PREPROCESSING FINAL SUMMARY
Student                   Rows=100   Columns=8     Missing=0   Duplicates=0  
Curriculum                Rows=20    Columns=9     Missing=0   Duplicates=0  
Activity                  Rows=60    Columns=12    Missing=0   Duplicates=0  
Assessment                Rows=30    Columns=15    Missing=0   Duplicates=0  
Teacher Observation       Rows=100   Columns=9     Missing=0   Duplicates=0  
Historical Learning Path  Rows=60    Columns=6     Missing=0   Duplicates=0  
AI Learning Path          Rows=11    Columns=14    Missing=0   Duplicates=0  
Preprocessing stage completed.
Data is ready for Feature Engineering.


In [147]:
import os

PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Processed directory created:", PROCESSED_PATH)

Processed directory created: ../data/processed


In [148]:
# Check all DataFrame variables currently available

import pandas as pd

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(name, "->", value.shape)

_ -> (10, 2)
student_df -> (100, 8)
curriculum_df -> (21, 9)
activity_df -> (60, 12)
assessment_df -> (30, 15)
teacher_df -> (100, 9)
historical_df -> (60, 6)
ai_learning_df -> (11, 14)
df -> (11, 14)
curriculum_clean -> (20, 9)
activity_clean -> (60, 12)
student_clean -> (100, 8)
assessment_clean -> (30, 15)
teacher_clean -> (100, 9)
historical_clean -> (60, 6)
ai_learning_clean -> (11, 14)
_31 -> (10, 2)
student_activity_features -> (15, 6)
completion_rate -> (15, 2)
assessment_features -> (15, 11)
student_features -> (15, 32)
teacher_features -> (100, 9)
historical_features -> (15, 9)
successful_outcomes -> (15, 2)
model_data -> (15, 29)
learning_path_features -> (15, 30)
learning_path_data -> (60, 35)
modeling_data -> (60, 35)
X -> (60, 33)
X_model -> (60, 30)


In [149]:
# ============================================================
# SAVE PREPROCESSED DATASETS
# ============================================================

import os

PROCESSED_PATH = "../data/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

student_clean.to_csv(
    f"{PROCESSED_PATH}/student_clean.csv",
    index=False
)

curriculum_clean.to_csv(
    f"{PROCESSED_PATH}/curriculum_clean.csv",
    index=False
)

activity_clean.to_csv(
    f"{PROCESSED_PATH}/activity_clean.csv",
    index=False
)

assessment_clean.to_csv(
    f"{PROCESSED_PATH}/assessment_clean.csv",
    index=False
)

teacher_clean.to_csv(
    f"{PROCESSED_PATH}/teacher_clean.csv",
    index=False
)

historical_clean.to_csv(
    f"{PROCESSED_PATH}/historical_clean.csv",
    index=False
)

ai_learning_clean.to_csv(
    f"{PROCESSED_PATH}/ai_learning_clean.csv",
    index=False
)

print("All preprocessed datasets saved successfully.")

All preprocessed datasets saved successfully.


In [150]:
# ============================================================
# VERIFY PROCESSED FILES
# ============================================================

print("=" * 70)
print("FILES IN PROCESSED DIRECTORY")
print("=" * 70)

for file in sorted(os.listdir(PROCESSED_PATH)):
    print(file)

FILES IN PROCESSED DIRECTORY
.DS_Store
.gitkeep
activity_clean.csv
ai_learning_clean.csv
assessment_clean.csv
curriculum_clean.csv
cv_prediction_results.csv
final_personalized_learning_paths.csv
historical_clean.csv
model_accuracy_comparison.png
model_comparison.csv
model_evaluation_metrics.csv
model_f1_comparison.png
model_ranking.csv
recommendation_test_results.csv
student_clean.csv
student_features.csv
student_learning_path_summary.csv
student_learning_path_validation.csv
teacher_clean.csv
validated_personalized_learning_paths.csv
